In [1]:
import sys
sys.path.append("../src/")
from constrained_likelihood_surrogates import *

In [ ]:

## FIRES ##


In [2]:
year = 2019
fire_seq = np.load('../data/fires/fireSize-' + str(year) + '.npy', 'r')
fire_seq = [int(val) for val in fire_seq]
# np.savetxt('fires-aus-2019.txt', fire_seq)

In [3]:
# Find lower cutoffs by minimising KS distance:

# ks_method = 'ave'
ks_method = 'sup'

model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
num_models = len(model_list)
year = 2019
val_seq = fire_seq
print('Year = ' + str(year) + ', N = ' + str(len(val_seq)) + ', min. = ' + str(min(val_seq)) + ', max. = ' + str(max(val_seq)) + ', mean = ' + str(np.mean(val_seq)))
lower_cutoff_hat_list_0 = [np.nan for i_model in range(num_models)]
lower_cutoff_hat_seq_list = []
ks_seq_list = []
for i_model in range(num_models):
    model = model_list[i_model]
    lower_cutoff_hat, ks, lower_cutoff_hat_seq, ks_seq = fit_lower_cutoff(val_seq, model=model, continuous=False, ks_method=ks_method)
    #ks_nan_pos_list = np.nonzero(np.isnan(ks_seq))[0].tolist()
    #print([i_year, i_model, lower_cutoff_hat, ks_nan_pos_list])
    lower_cutoff_hat_list_0[i_model] = lower_cutoff_hat
    lower_cutoff_hat_seq_list = lower_cutoff_hat_seq_list + [lower_cutoff_hat_seq]
    ks_seq_list = ks_seq_list + [ks_seq]
print('x_min = ' + str(lower_cutoff_hat_list_0))
N_null_list_0 = [len([val for val in val_seq if val >= lower_cutoff_hat]) for lower_cutoff_hat in lower_cutoff_hat_list_0]
print('N = ' + str(N_null_list_0))

Year = 2019, N = 8692, min. = 1, max. = 12446, mean = 29.398872526461112
x_min = [40.5, 32.5, 1241.5, 4806.5, 2820.5]
N = [417, 475, 33, 13, 17]


In [4]:
#Load previously saved lower cutoffs:

lower_cutoff_type = 'mod-ks-sup'#Lower cutoff found by minimising KS distance for that model
#Calculating KS distance as supremum of absolute difference
ks_method = 'sup'
ecdf_method = 'max'

year = 2019

lower_cutoff_hat_list_0 = [40.5, 32.5, 1241.5, 3495.5, 2820.5]
model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
N_null_list_0 = [417, 475, 33, 15, 17]

In [5]:
# Determining REJECTION RATE from constrained and typical surrogates for FIRE DATA FROM 2019:
# 
# Use several types of statistics as test statistics
# 
# Considering constrained and typical surrogates
# 
# 
# Model-free statistics:
free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
num_free_stats = len(free_stat_list)
# Model-dependent statistics:
# depe_stat_list = [ks_stat, ad_stat, cvm_stat]
depe_stat_list = [ks_stat, kuiper_stat, ad_stat, cvm_stat, zk_stat, za_stat, zc_stat]
num_depe_stats = len(depe_stat_list)

test_type = 'rejection'
lower_cutoff = lower_cutoff_hat_list_0[0]#lower cutoff which minimises KS distance for maximum likelihood power-law model
# lower_cutoff = lower_cutoff_hat_list_0[1]#lower cutoff which minimises KS distance for maximum likelihood lognormal model
val_seq_full = [val for val in fire_seq if (val >= lower_cutoff)]
N_max = len(val_seq_full)#Number of data not less than lower cutoff under power-law model
log_N_list = np.linspace(np.log(4), np.log(N_max), 9)
N_list = np.round(np.exp(log_N_list))
N_list = [int(N) for N in N_list]
print('N_list = ' + str(N_list))
for N in N_list:
    start_time = timer()
    num_trans = N*(int(np.ceil(np.log2(N*1024))))
    
    num_surr = 9
    num_tests = 10**1
    
    a = lower_cutoff
    b = 9
    
    print('N = ' + str(N) + ', lower cut-off = ' + str(lower_cutoff))
    
    print(str(num_tests) + ' tests, each with ' + str(num_surr) + ' constrained, typical surrogates, each using ' + str(num_trans) + ' transitions')
    
    # model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
    model_list = ['powerlaw', 'lognorm']
    num_models = len(model_list)
    
    num_methods = 2#Based on constrained, typical

    save_str_0 = 'quant_free-depe' + '_' + test_type + '_xmin-' + str(lower_cutoff) + '_N-' + str(N) + '_ntra-' + str(num_trans) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_nmod-' + str(num_models) + '_nfs-' + str(num_free_stats) + '_nds-' + str(num_depe_stats)
    
    process_str = '_fires-2019'
    
    free_quantile_list_list = np.full((num_methods, num_models, num_tests, num_free_stats), np.nan)#Array of nans
    depe_quantile_list_list = np.full((num_methods, num_models, num_tests, num_depe_stats), np.nan)#Array of nans

    for i_test in range(num_tests):
        #Generate time series and calculate ks distance:
        val_seq = random.sample(val_seq_full, N)
        lower_cutoff_hat = lower_cutoff
        free_stat_val_list = [stat(val_seq) for stat in free_stat_list]
        for i_model in range(num_models):
            model = model_list[i_model]
            param_hat_m, lp_seq_m = fit_model(val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)#m for model
            val_seq_sorted = sorted(val_seq)
            emp_cdf_with_rep = rankdata(val_seq_sorted, method='max')/len(val_seq_sorted)
            cdf_fun = return_cdf_func(param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
            exp_cdf_with_rep = cdf_fun(val_seq_sorted)
            depe_stat_val_list = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
            
            c_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Constrained
            t_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Typical
            
            c_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Constrained
            t_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Typical
            
            surr_m_val_seq = val_seq
            surr_m_val_seq_list = []
            for i_surr in range(num_surr):
                method = 'constrained'
                surr_m_val_seq = gen_surrogate(surr_m_val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                random.shuffle(surr_m_val_seq)
                surr_m_val_seq_list = surr_m_val_seq_list + [surr_m_val_seq]

            for i_surr in range(num_surr):
                method = 'constrained'
                surr_m_val_seq = surr_m_val_seq_list[i_surr]
                surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                c_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                c_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m

            for i_surr in range(num_surr):
                method = 'typical'
                surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                t_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                t_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m

            for i_free_stat in range(num_free_stats):
                #Calculate values of discriminating statistic (model-free statistics) for observed sequence and surrogate sequences:
                obs_stat = free_stat_val_list[i_free_stat]
                abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                if (abs_obs_stat == np.inf):
                    abs_obs_stat = 0
                obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                for i_method in range(num_methods):
                    if (i_method == 0):#Constrained surrogates
                        stat_val_surr_m_list = c_surr_m_free_list_list[:, i_free_stat]
                    elif (i_method == 1):#Typical surrogates
                        stat_val_surr_m_list = t_surr_m_free_list_list[:, i_free_stat]
                    stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                    #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                    rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                    rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                    r = random.randint(rankMin, rankMax)
                    q = (r - 0.5)/(num_surr + 1)
                    free_quantile_list_list[i_method, i_model, i_test, i_free_stat] = q
                    if (np.isnan(q)):
                        raise Exception('Calculated quantile q is not a number.')

            for i_depe_stat in range(num_depe_stats):
                #Calculate values of discriminating statistic (model-dependent statistics) for observed sequence and surrogate sequences:
                obs_stat = depe_stat_val_list[i_depe_stat]
                abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                if (abs_obs_stat == np.inf):
                    abs_obs_stat = 0
                obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                for i_method in range(num_methods):
                    if (i_method == 0):#Constrained surrogates
                        stat_val_surr_m_list = c_surr_m_depe_list_list[:, i_depe_stat]
                    elif (i_method == 1):#Typical surrogates
                        stat_val_surr_m_list = t_surr_m_depe_list_list[:, i_depe_stat]
                    stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                    #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                    rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                    rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                    r = random.randint(rankMin, rankMax)
                    q = (r - 0.5)/(num_surr + 1)
                    depe_quantile_list_list[i_method, i_model, i_test, i_depe_stat] = q
                    if (np.isnan(q)):
                        raise Exception('Calculated quantile q is not a number.')

    end_time = timer()
    total_time = end_time - start_time
    print('N=' + str(N) + ' took ' + str(total_time) + 'sec.') # Time in seconds, e.g. 5.38091952400282
        
    save_str = save_str_0 + process_str

    free_stat_name_list = [stat.__name__ for stat in free_stat_list]
    depe_stat_name_list = [stat.__name__ for stat in depe_stat_list]

    save_dict = {'free_quantile_list_list':free_quantile_list_list.tolist(),
                 'depe_quantile_list_list':depe_quantile_list_list.tolist(),
                 'test_type':test_type,
                 'lower_cutoff':lower_cutoff,
                 'N':N,
                 'num_trans':num_trans,
                 'num_surr':num_surr,
                 'num_tests':num_tests,
                 'num_methods':num_methods,
                 'process_str':process_str,
                 'model_list':model_list,
                 'save_str':save_str,
                 'total_time':total_time,
                 'free_stat_name_list':free_stat_name_list,
                 'depe_stat_name_list':depe_stat_name_list,
                 'fire_seq':fire_seq,
                 'val_seq_full':val_seq_full,
    }
    
    save('./results/hyp-test/fires/' + save_str, save_dict)

N_list = [4, 7, 13, 23, 41, 73, 131, 233, 417]
N = 4, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 48 transitions
N=4 took 2.8335038000004715sec.
N = 7, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 91 transitions
N=7 took 2.7960345999999845sec.
N = 13, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 182 transitions
N=13 took 3.2537941000009596sec.
N = 23, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 345 transitions
N=23 took 3.789399699999194sec.
N = 41, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 656 transitions
N=41 took 5.634846299999481sec.
N = 73, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 1241 transitions
N=73 took 7.554573100000198sec.
N = 131, lower cut-off = 40.5
10 tests, each with 9 constrained, typical surrogates, each using 2

In [6]:
# For fires data from 2019,
# Investigating mean and variance of some estimates of statistics, using data below and above the upper cutoff and downsampling

# Checking distribution of statistics under several types of surrogates:
# Constrained, typical, true, original, bootstrapped
# 
sp = '  '#Making indentations to increase readability 
test_type = 'statistics'
num_surr = 10**1

print(str(num_surr) + ' constrained, typical surrogates')

null_model_list = ['powerlaw', 'lognorm']
k_null_list = [1, 2]#Power-law, lognormal
# null_model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
# k_null_list = [1, 2, 1, 2, 1]#Power-law, lognormal

num_null_models = len(null_model_list)

method_list = ['constrained', 'typical', 'original', 'bootstrap']

num_methods = 4#Based on constrained, typical, original, bootstrap

stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
stat_name_list = [stat.__name__ for stat in stat_list]
num_stats = len(stat_list)

# lower_cutoff_hat = lower_cutoff_hat_list_0[0]#lower cutoff which minimises KS distance for maximum likelihood power-law model
lower_cutoff_hat = lower_cutoff_hat_list_0[1]#lower cutoff which minimises KS distance for maximum likelihood lognormal model
val_seq = [val for val in fire_seq if (val >= lower_cutoff_hat)]
N = len(val_seq)#Original length
log_Ns_list = np.linspace(np.log(4), np.log(N), 9)
Ns_list = np.round(np.exp(log_Ns_list))
Ns_list = [int(Ns) for Ns in Ns_list]
Ns_list.reverse()
num_Ns = len(Ns_list)
print('Ns_list = ' + str(Ns_list))

lower_cutoff_type = lower_cutoff_hat

mean_stat_list_list_list = np.full((num_null_models, num_methods, num_Ns, num_stats), np.nan)#Array of nans: num_null_models x num_methods x numnum_stats
var_stat_list_list_list = np.full((num_null_models, num_methods, num_Ns, num_stats), np.nan)#Array of nans: num_null_models x num_methods x num_stats

save_str_0 = test_type + '_nsur-' + str(num_surr) + '_ntyp-' + str(num_methods) + '_nmod-' + str(num_null_models) + '_' + str(num_Ns) + 'Ns' + '-' + str(Ns_list[0]) + '-' + str(Ns_list[-1]) + '_nstats-' + str(num_stats)
data_str = '_fires' + '_year-' + str(year) + '_lower-cutoff-type-' + str(lower_cutoff_type) + '_above-xmin-only'
save_str = save_str_0 + data_str

N_null_list = []
num_trans_list = []

NLL_list = []
AIC_list = []
BIC_list = []

start_time = timer()

N_null_list = []#Original length above lower cutoff
param_hat_null_list = []#List of max. likelihood parameters for each null model
lp_seq_null_list = []#List of max. likelihood log probability values for each null model
val_seq_null_list = []#List of time series for each null model
null_model_time_list = []#List of time taken for each null model
for i_null_model in range(num_null_models):
    null_model = null_model_list[i_null_model]
    lower_cutoff_hat_null = lower_cutoff_hat
    val_seq_null = val_seq
    val_seq_null_list = val_seq_null_list + [val_seq_null]
    N_null = len(val_seq_null)
    N_null_list = N_null_list + [N_null]
    param_hat_null, lp_seq_null = fit_model(val_seq_null, lower_cutoff_hat=lower_cutoff_hat_null, model=null_model)
    param_hat_null_list = param_hat_null_list + [param_hat_null]
    lp_seq_null_list = lp_seq_null_list + [lp_seq_null]
    k = k_null_list[i_null_model]
    NLL = -sum(lp_seq_null)#Negative log-likelihood
    NLL_list = NLL_list + [NLL]
    BIC = 2*NLL + np.log(N_null)*k
    BIC_list = BIC_list + [BIC]
    AIC = 2*NLL + 2*k
    AIC_list = AIC_list + [AIC]
for i_null_model in range(num_null_models):
    start_time_null_model = timer()
    null_model = null_model_list[i_null_model]
    lower_cutoff_hat_null = lower_cutoff_hat
    val_seq_null = val_seq_null_list[i_null_model]
    N_null = len(val_seq_null)
    print(1*sp + 'Null model for largest N_null = ' + str(N_null) + ' data is ' + null_model + ' with lower cutoff ' + str(lower_cutoff_hat_null))
    if (N_null > 0):
        num_trans = N_null*(int(np.ceil(np.log2(N_null*1024))))
    else:
        num_trans = 0
        print(1*sp + 'Skipping ' + null_model + ' null model')
        continue
    num_trans_list = num_trans_list + [num_trans]
    param_hat_null = param_hat_null_list[i_null_model]
    lp_seq_null = lp_seq_null_list[i_null_model]
    c_surr_null_val_seq = val_seq_null

    c_surr_null_stat_val_list_list_list = np.zeros(shape=(num_surr, num_Ns, num_stats))#Constrained
    t_surr_null_stat_val_list_list_list = np.zeros(shape=(num_surr, num_Ns, num_stats))#Typical
    tr_surr_null_stat_val_list_list_list = np.zeros(shape=(num_surr, num_Ns, num_stats))#True
    o_surr_null_stat_val_list_list_list = np.zeros(shape=(num_surr, num_Ns, num_stats))#Original
    b_surr_null_stat_val_list_list_list = np.zeros(shape=(num_surr, num_Ns, num_stats))#Bootstrapped

    surr_null_val_seq_upper = val_seq_null
    surr_null_val_seq_list = []
    for i_surr in range(num_surr):
        method = 'constrained'
        surr_null_val_seq_upper = gen_surrogate(surr_null_val_seq_upper, model=null_model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat_null, param_hat=param_hat_null)
        random.shuffle(surr_null_val_seq_upper)
        surr_null_val_seq = surr_null_val_seq_upper.copy()
        random.shuffle(surr_null_val_seq)
        surr_null_val_seq_list = surr_null_val_seq_list + [surr_null_val_seq]

    for i_surr in range(num_surr):
        method = 'constrained'
        surr_null_val_seq = surr_null_val_seq_list[i_surr]
        surr_null_val_seq_s = surr_null_val_seq
        for i_Ns in range(num_Ns):
            Ns = Ns_list[i_Ns]
            surr_null_val_seq_s = random.sample(surr_null_val_seq_s, Ns)
            surr_null_stat_val_list = [stat(surr_null_val_seq_s) for stat in stat_list]
            c_surr_null_stat_val_list_list_list[i_surr, i_Ns, :] = surr_null_stat_val_list

    for i_surr in range(num_surr):
        method = 'typical'
        N_null_2 = binom.rvs(N, N_null/N)
        surr_null_val_seq = gen_data(null_model, N=N_null_2, lower_cutoff=lower_cutoff_hat_null, param=param_hat_null)
        random.shuffle(surr_null_val_seq)
        surr_null_val_seq_s = surr_null_val_seq
        for i_Ns in range(num_Ns):
            Ns = Ns_list[i_Ns]
            surr_null_val_seq_s = random.sample(surr_null_val_seq_s, Ns)
            surr_null_stat_val_list = [stat(surr_null_val_seq_s) for stat in stat_list]
            t_surr_null_stat_val_list_list_list[i_surr, i_Ns, :] = surr_null_stat_val_list

    for i_surr in range(num_surr):
        method = 'original'
        surr_null_val_seq = val_seq
        surr_null_val_seq_s = surr_null_val_seq
        for i_Ns in range(num_Ns):
            Ns = Ns_list[i_Ns]
            surr_null_val_seq_s = random.sample(surr_null_val_seq_s, Ns)
            surr_null_stat_val_list = [stat(surr_null_val_seq_s) for stat in stat_list]
            o_surr_null_stat_val_list_list_list[i_surr, i_Ns, :] = surr_null_stat_val_list

    for i_surr in range(num_surr):
        method = 'bootstrap'
        surr_null_val_seq = bootstrap(val_seq, N=N)
        surr_null_val_seq_s = surr_null_val_seq
        for i_Ns in range(num_Ns):
            Ns = Ns_list[i_Ns]
            surr_null_val_seq_s = random.sample(surr_null_val_seq_s, Ns)
            surr_null_stat_val_list = [stat(surr_null_val_seq_s) for stat in stat_list]
            b_surr_null_stat_val_list_list_list[i_surr, i_Ns, :] = surr_null_stat_val_list

    for i_method in range(num_methods):
        method = method_list[i_method]
        if (method == 'constrained'):#Constrained surrogates
            stat_val_surr_null_list_list_list = c_surr_null_stat_val_list_list_list
        elif (method == 'typical'):#Typical surrogates
            stat_val_surr_null_list_list_list = t_surr_null_stat_val_list_list_list
        elif (method == 'original'):#Original sequence
            stat_val_surr_null_list_list_list = o_surr_null_stat_val_list_list_list
        elif (method == 'bootstrap'):#Bootstrapping
            stat_val_surr_null_list_list_list = b_surr_null_stat_val_list_list_list
        else:
            raise Exception('I am not prepared for this surrogate method.')
        mean_stat_list_list_list[i_null_model, i_method, :, :] = np.mean(stat_val_surr_null_list_list_list, axis=0)
        var_stat_list_list_list[i_null_model, i_method, :, :]  = np.var(stat_val_surr_null_list_list_list, axis=0)
        
    end_time_null_model = timer()
    total_time_null_model = end_time_null_model - start_time_null_model
    null_model_time_list = null_model_time_list + [total_time_null_model]
    print('Null model ' + null_model + ' took ' + str(total_time_null_model) + 'sec.')

end_time = timer()
total_time = end_time - start_time

save_dict = {'mean_stat_list_list_list':mean_stat_list_list_list.tolist(),
             'var_stat_list_list_list':var_stat_list_list_list.tolist(),
             'val_seq':val_seq,
             'test_type':test_type,
             'lower_cutoff_type':lower_cutoff_type,
             'lower_cutoff_hat':lower_cutoff_hat,
             'N':N,
             'Ns_list':Ns_list,
             'num_trans_list':num_trans_list,
             'num_surr':num_surr,
             'method_list':method_list,
             'num_methods':num_methods,
             'null_model_list':null_model_list,
             'data_str':data_str,
             'k_null_list':k_null_list,
             'NLL_list':NLL_list,
             'BIC_list':BIC_list,
             'AIC_list':AIC_list,
             'param_hat_null_list':param_hat_null_list,
             'save_str':save_str,
             'stat_name_list':stat_name_list,
             'num_stats':num_stats,
             'total_time':total_time,
}

save('./results/est-stat/' + save_str, save_dict)

10 constrained, typical surrogates
Ns_list = [475, 261, 144, 79, 44, 24, 13, 7, 4]
  Null model for largest N_null = 475 data is powerlaw with lower cutoff 32.5
Null model powerlaw took 4.74430989999928sec.
  Null model for largest N_null = 475 data is lognorm with lower cutoff 32.5
Null model lognorm took 6.230079399998431sec.
